<div style="
    background-color: #3b8132; 
    padding: 10px; 
    border-radius: 5px;
    text-align: center;
    font-weight: bold;
    font-size: 24px;
    color: white;
">
 Библиотеки
</div>

In [ ]:
import pandas as pd
from datetime import datetime, timezone, timedelta
import csv
import gzip
import json
import random
import requests
from collections import defaultdict, Counter
from functools import lru_cache
from pathlib import Path
import plotly.express as px
from datetime import timedelta
import geopandas as gpd
from shapely.geometry import Point

DATA_FILE = Path("collector_all.csv") # файл с объединенными данными
DISTRICTS_FILE = Path("data/clean_spb.geojson") # файл с границами районов
MOSCOW = timezone(timedelta(hours=3))

<div style="
    background-color: #3b8132; 
    padding: 10px; 
    border-radius: 5px;
    text-align: center;
    font-weight: bold;
    font-size: 24px;
    color: white;
">
 Данные о технике

</div>

In [2]:
response = requests.get("https://ut.kb.gov.spb.ru/info.json", timeout=20)
response.raise_for_status()
meta_data = response.json()

did_job = {}
meta_by_id = {}
for i in meta_data['result']:
    did = i.get('did')
    if did:
        did_job[str(did)] = i.get('job', '')
        meta_by_id[str(did)] = i

car_Fun = [""] * 17
car_Fun[1] = "Погрузчики"
car_Fun[2] = "Самосвалы и МСК"
car_Fun[3] = "Мусоровозы"
car_Fun[4] = "Распределители твердых реагентов"
car_Fun[5] = "Распределители твердых реагентов с увлажнением"
car_Fun[6] = "Поливомоечное оборудование"
car_Fun[7] = "Подметально-уборочное оборудование (механическое)"
car_Fun[8] = "Вакуумное оборудование"
car_Fun[9] = "Щеточное оборудование (на автомобильном шасси)"
car_Fun[10] = "Плужное оборудование (на автомобильном шасси)"
car_Fun[11] = "Бульдозеры"
car_Fun[12] = "Распределители жидких реагентов"
car_Fun[13] = "Тягач (для уборочной техники)"
car_Fun[15] = "Контроль"
car_Fun[16] = "Ручная уборка"

In [3]:
def get_function(m):
    vid = str(m['id'])
    t = meta_by_id.get(vid)
    if t is None or not m.get('info'):
        return None
    if pd.isna(t.get('job')):
        return "Человек"

    info = m['info']
    if isinstance(info, str):
        import ast
        info = ast.literal_eval(info)
    key = int(info[2])

    for i in range(1, 5):
        mask = t.get(f'F{i}_MASK')
        level = t.get(f'F{i}_AL')
        func_id = t.get(f'F{i}_ID')
        if pd.notna(mask) and pd.notna(level) and pd.notna(func_id) and float(mask) != 0:
            if (key & int(mask)) / float(mask) == float(level):
                return car_Fun[int(func_id)]
    if (8 & key) / 8 == 1:
        return "Холостой ход"
    return "Стоянка"

<div style="
    background-color: #3b8132; 
    padding: 10px; 
    border-radius: 5px;
    text-align: center;
    font-weight: bold;
    font-size: 24px;
    color: white;
">
 Районы

</div>

In [ ]:
districts = gpd.read_file(DISTRICTS_FILE).to_crs(epsg=4326)
name_col = 'Name'

@lru_cache(maxsize=20000)
def get_district(lat, lon):
    point = Point(lon, lat)
    for index in districts.sindex.query(point, predicate='intersects'):
        district = districts.iloc[int(index)]
        if district.geometry.covers(point):
            return str(district[name_col])
    return None

<div style="
    background-color: #3b8132; 
    padding: 10px; 
    border-radius: 5px;
    text-align: center;
    font-weight: bold;
    font-size: 24px;
    color: white;
">
 Основная статистика

</div>

In [5]:
def read_messages():
    opener = gzip.open if DATA_FILE.suffix == '.gz' else open
    with opener(DATA_FILE, 'rt', encoding='utf-8-sig', newline='') as f:
        for row in csv.reader(f):
            try:
                m = json.loads(row[0])
                if isinstance(m, dict) and all(x in m for x in ('id', 'ut', 'loc')):
                    yield m
            except (ValueError, IndexError, TypeError):
                continue


data = {}                  # id -> [первое время, последнее время, число записей]
locs = {}                  # id -> первая координата
active = set()
hour_ids = defaultdict(set)
hour_moving = defaultdict(set)
hour_first_loc = {}
day_ids = defaultdict(set)
day_obs = Counter()
areas = defaultdict(set)
district_hour = defaultdict(set)
func_cars = defaultdict(set)
district_func_cars = defaultdict(set)

for m in read_messages():
    vid, ut, loc = str(m['id']), float(m['ut']), tuple(m['loc'])
    t = datetime.fromtimestamp(ut, timezone.utc).astimezone(MOSCOW)
    hour = t.replace(minute=0, second=0, microsecond=0)

    if vid not in data:
        data[vid] = [ut, ut, 0]
        locs[vid] = loc
    data[vid][0] = min(data[vid][0], ut)
    data[vid][1] = max(data[vid][1], ut)
    data[vid][2] += 1
    if loc != locs[vid]:
        active.add(vid)

    day_ids[t.date()].add(vid)
    day_obs[t.date()] += 1
    hour_ids[hour].add(vid)
    key = (hour, vid)
    if key not in hour_first_loc:
        hour_first_loc[key] = loc
    elif loc != hour_first_loc[key]:
        hour_moving[hour].add(vid)

    district = get_district(round(loc[0], 5), round(loc[1], 5))
    if district:
        areas[vid].add(district)
        district_hour[(hour, district)].add(vid)

    func = get_function(m)
    if func:
        func_cars[func].add(vid)
        if district:
            district_func_cars[(district, func)].add(vid)

func_counts = {func: len(cars) for func, cars in func_cars.items()}

active = list(active)
print(f"Из {len(locs)} машин {len(active)} активные")

by_type = {}
for vid in active:
    t = did_job.get(vid)
    by_type[t] = by_type.get(t, 0) + 1
print(by_type)

Из 2103 машин 2075 активные
{'Комбинированные мaшины': 747, 'Погрузочная техника': 475, 'Комбинированные машины': 444, 'Самосвалы и МСК': 61, 'Легковая (для уборочной техники)': 71, 'Механические подметально-уборочные машины (в т.ч. прицепы и ВС)': 28, 'Вакуумные уборочные машины': 167, 'Мусоровозы': 60, 'Снегоочистители': 10, 'Тягач (для уборочной техники)': 4, None: 6, 'Бульдозеры': 2}


In [6]:
first_ut = min(points[0] for points in data.values())
last_ut = max(points[1] for points in data.values())
first_time = datetime.fromtimestamp(first_ut, timezone.utc).astimezone(MOSCOW)
last_time = datetime.fromtimestamp(last_ut, timezone.utc).astimezone(MOSCOW)
all_hours = pd.date_range(
    first_time.replace(minute=0, second=0, microsecond=0),
    last_time.replace(minute=0, second=0, microsecond=0),
    freq='h'
)

df_summary = pd.DataFrame({
    'Показатель': ['Начало наблюдений', 'Конец наблюдений', 'Обработано записей',
                   'Уникальных машин', 'Меняли координаты', 'Часов без записей'],
    'Значение': [str(first_time), str(last_time), sum(x[2] for x in data.values()),
                 len(data), len(active), len(all_hours) - len(hour_ids)]
})
display(df_summary)

,Показатель,Значение
0,Начало наблюдений,2026-10-17 23:38:44+03:00
1,Конец наблюдений,2026-10-24 17:59:59+03:00
2,Обработано записей,25358514
3,Уникальных машин,2103
4,Меняли координаты,2075
5,Часов без записей,4


In [7]:
df_type = pd.DataFrame(list(by_type.items()), columns=['Тип', 'Количество машин'])
df_type = df_type.sort_values('Количество машин', ascending=False)
fig = px.bar(df_type, x='Тип', y='Количество машин', title='Типы активной техники')
fig.show()

<div style="
    background-color: #3b8132; 
    padding: 10px; 
    border-radius: 5px;
    text-align: center;
    font-weight: bold;
    font-size: 24px;
    color: white;
">
 Активность по времени

</div>

In [8]:
f = []
for t in sorted(hour_ids):
    f.append({'Время': t, 'Количество': len(hour_ids[t]), 'Меняли координаты': len(hour_moving[t])})
df_hourly = pd.DataFrame(f)

fig = px.line(df_hourly, x='Время', y=['Количество', 'Меняли координаты'],
              title='Активность техники по часам')
fig.show()

df_hourly['Час'] = df_hourly['Время'].dt.hour
df_hourly['День недели'] = df_hourly['Время'].dt.dayofweek

fig = px.bar(df_hourly.groupby('Час', as_index=False)['Количество'].mean(),
             x='Час', y='Количество', title='Среднее число машин по часам суток')
fig.show()

fig = px.bar(df_hourly.groupby('День недели', as_index=False)['Количество'].mean(),
             x='День недели', y='Количество', title='Среднее число машин по дням недели (0 — понедельник)')
fig.show()

In [9]:
df_daily = pd.DataFrame([
    {'Дата': str(day), 'Машин': len(day_ids[day]), 'Наблюдений': day_obs[day]}
    for day in sorted(day_ids)
])

fig = px.bar(df_daily, x='Дата', y='Машин', title='Уникальные машины по дням')
fig.show()

fig = px.bar(df_daily, x='Дата', y='Наблюдений', title='Количество записей по дням')
fig.show()

<div style="
    background-color: #3b8132; 
    padding: 10px; 
    border-radius: 5px;
    text-align: center;
    font-weight: bold;
    font-size: 24px;
    color: white;
">
 Распределение по районам

</div>

In [10]:
areas_count = {}
for vid, districts_for_car in areas.items():
    for district in districts_for_car:
        if district in areas_count:
            areas_count[district] += 1
        else:
            areas_count[district] = 1

areas_count = dict(sorted(areas_count.items(), key=lambda x: x[1], reverse=True))
print(areas_count)

fig = px.bar(x=list(areas_count), y=list(areas_count.values()),
             labels={'x': 'Район', 'y': 'Количество машин'},
             title='Уникальные машины по районам')
fig.show()

f = [{'Время': hour, 'Район': district, 'Количество': len(ids)}
     for (hour, district), ids in district_hour.items()]
df_district_hour = pd.DataFrame(f)
display(df_district_hour.head())

{'Московский район': 1068, 'Фрунзенский район': 1004, 'Красногвардейский район': 945, 'Невский район': 714, 'Калининский район': 603, 'Центральный район': 585, 'Выборгский район': 583, 'Кировский район': 539, 'Адмиралтейский район': 491, 'Пушкинский район': 481, 'Василеостровский район': 450, 'Приморский район': 446, 'Курортный район': 437, 'Красносельский район': 370, 'Петроградский район': 345, 'Колпинский район': 321, 'Петродворцовый район': 226, 'Кронштадтский район': 167}


,Время,Район,Количество
0,2026-10-17 23:00:00+03:00,Московский район,2
1,2026-10-17 23:00:00+03:00,Приморский район,6
2,2026-10-17 23:00:00+03:00,Выборгский район,9
3,2026-10-17 23:00:00+03:00,Красногвардейский район,1
4,2026-10-17 23:00:00+03:00,Невский район,5


In [11]:
f = [
    {'Район': str(district), 'Час': hour.hour,
     'Количество': len(district_hour.get((hour, str(district)), set()))}
    for hour in sorted(hour_ids)
    for district in districts[name_col]
]
df_heat = pd.DataFrame(f)
heatmap = df_heat.pivot_table(index='Район', columns='Час', values='Количество', aggfunc='mean')

fig = px.imshow(
    heatmap,
    aspect='auto',
    labels={'x': 'Час суток', 'y': 'Район', 'color': 'Среднее число машин'},
    title='Среднее число наблюдавшихся машин по районам и часам'
)
fig.update_layout(height=650)
fig.show()

<div style="
    background-color: #3b8132; 
    padding: 10px; 
    border-radius: 5px;
    text-align: center;
    font-weight: bold;
    font-size: 24px;
    color: white;
">
 Операции

</div>

In [12]:
df_funcs = pd.DataFrame(func_counts.items(), columns=['Операция', 'Количество машин'])
df_funcs = df_funcs.sort_values('Количество машин', ascending=False)
display(df_funcs)
fig = px.bar(df_funcs, x='Операция', y='Количество машин', title='Распределение машин по операциям')
fig.show()

,Операция,Количество машин
1,Стоянка,1698
0,Холостой ход,1000
9,Щеточное оборудование (на автомобильном шасси),312
2,Погрузчики,192
8,Вакуумное оборудование,189
11,Поливомоечное оборудование,180
7,Самосвалы и МСК,179
3,Бульдозеры,120
12,Распределители твердых реагентов,120
6,Распределители жидких реагентов,115


In [13]:
df_district_funcs = pd.DataFrame([
    {'Район': district, 'Операция': func, 'Количество машин': len(cars)}
    for (district, func), cars in district_func_cars.items()
])

fig = px.bar(df_district_funcs, x='Район', y='Количество машин', color='Операция',
             barmode='stack', title='Распределение техники по операциям и районам')
fig.update_layout(xaxis_tickangle=-45, height=600)
fig.show()

<div style="
    background-color: #3b8132; 
    padding: 10px; 
    border-radius: 5px;
    text-align: center;
    font-weight: bold;
    font-size: 24px;
    color: white;
">
 Карта пяти машин

</div>

In [17]:
start = datetime(2026, 10, 22, 8, 0, tzinfo=MOSCOW)
finish = start + timedelta(hours=12)

start_ut = start.timestamp()
finish_ut = finish.timestamp()

counts = Counter()

for m in read_messages():
    if start_ut <= float(m['ut']) < finish_ut:
        counts[str(m['id'])] += 1

available = [vid for vid, n in counts.items() if n >= 2]
ids = random.sample(available, min(5, len(available)))

print('Выбранные машины:', ids)

step = {vid: max(1, (counts[vid] + 599) // 600) for vid in ids}
seen = defaultdict(int)
all_points = []

for m in read_messages():
    vid = str(m['id'])
    ut = float(m['ut'])

    if vid not in ids or not (start_ut <= ut < finish_ut):
        continue

    seen[vid] += 1

    if (seen[vid] - 1) % step[vid] == 0:
        loc = m['loc']
        all_points.append({
            'id': vid,
            'lat': loc[0],
            'lon': loc[1],
            'ut': ut
        })

df = pd.DataFrame(all_points)

if not df.empty:
    df_sample = df.sort_values(['id', 'ut']).copy()

    # разделяем маршрут, если между точками прошло более 20 минут
    gap = df_sample.groupby('id')['ut'].diff() > 20 * 60
    df_sample['segment'] = (
        df_sample['id'] + '_' +
        gap.groupby(df_sample['id']).cumsum().astype(str)
    )

    fig = px.line_map(
        df_sample,
        lat='lat',
        lon='lon',
        color='id',
        line_group='segment',
        hover_name='id',
        hover_data={'ut': True},
        center={'lat': 59.94, 'lon': 30.32},
        zoom=10,
        height=600,
        map_style='carto-positron',
        title='Траектории 5 машин за 12 часов'
    )

    fig.update_traces(mode='lines+markers')

    fig.update_layout(
        margin={'l': 0, 'r': 0, 't': 30, 'b': 0}
    )

    fig.show()
else:
    print('За выбранный период данных нет')


Выбранные машины: ['301347', '800327487', '68208894', '300918', '6871']
